In [0]:
!pip install kaggle

In [0]:
%restart_python 

In [0]:
dbutils.library.restartPython() 

In [0]:
import os

os.environ["KAGGLE_USERNAME"] = "sangeethaignas"
os.environ["KAGGLE_KEY"] = "026c6676305f2f80290441834c8cc40c"

print("Kaggle credentials configured!")

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce
""")

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.ecommerce_data
""")

In [0]:
%sh
cd /tmp
kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store


In [0]:
!ls /tmp

In [0]:
%sh
mkdir -p /Volumes/workspace/ecommerce/ecommerce_data/
unzip /tmp/ecommerce-behavior-data-from-multi-category-store.zip -d /Volumes/workspace/ecommerce/ecommerce_data/


In [0]:
%restart_python

In [0]:
nov_df = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")

In [0]:
oct_df = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv",header=True,inferSchema=True)

In [0]:
print(f"October 2019 - Total Events: {df.count():,}")
print("\n" + "="*60)
print("SCHEMA:")
print("="*60)
oct_df.printSchema()

In [0]:
print("\n" + "="*60)
print("SAMPLE DATA (First 5 rows):")
print("="*60)
oct_df.show(5, truncate=True)
#-----------------------------------------------
print("__________________________________")
display(oct_df.limit(5))


In [0]:
# Load data
events = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv", header=True, inferSchema=True)

# Basic operations
events.select("event_type", "price").show(10)
events.filter("price > 100").count()
events.groupBy("event_type").count().show()
top_brands = events.groupBy("brand").count().orderBy("count", ascending=False).limit(5)
display(top_brands)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Top 5 products by revenue
revenue = events.filter(F.col("event_type") == "purchase") \
    .groupBy("product_id") \
    .agg(F.sum("price").alias("revenue")) \
    .orderBy(F.desc("revenue")).limit(5)

# Running total per user
window = Window.partitionBy("user_id").orderBy("event_time")
events.withColumn("cumulative_events", F.count("*").over(window))

# Conversion rate by category
events.groupBy("category_code") \
    .pivot("event_type") \
    .count() \
    .withColumn(
        "conversion_rate",
        F.col("purchase") / F.col("view") * 100
    )